
# Balancing chemical equations with SMT solvers

In this notebook, we will
look at SMT solvers and see how they can be used to formalize and solve
systems of equations. Particularly, we will show how we can solve equations
to balance chemical reactions. We will write a program that can balance more
reactions to introduce you to Z3, a state-of-the-art SMT solver,
and its capabilities.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Integers

Let's use Z3 to solve problems involving integers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Int('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()


Let's try an example with multiple equations. **Replace lines in the code below** to find a solution to the following system of equations:

$$x + 4y = 20$$
$$2x + 3y = 10$$


In [ ]:

# Initialize Z3 solver
s = Solver()

# Initialize variables

x = Real('x')
y = Real('y')

s.add( x + 4*y == 20 ) # add the first equation
s.add( False ) # REPLACE THIS LINE

showSolver( s ) # view the equations


In [ ]:
print( s.check() ) # check if solution exists

In [ ]:
print( s.model() ) # output solution


## Reaction Balancing using Z3

Recall that in the previous notebook, we wrote a program that balances the following reaction:

$$H_2 + O_2 \rightarrow  H_2O$$

Let's see how we can improve on this to make a program that can balance more reactions. 



### Representing Chemical Reactions

To write this program, we need a way to represent our reaction in python. 
To do this, we will use a dictionary to store each half of the reaction. 

In our example reaction, we have two reactants and one product, which we can represent as shown below.
Note that for each compound, we also need to know how much of each element is in it.


In [ ]:

reactants = {
  'H2': {'H': 2},
  'O2': {'O': 2}
}

products = {
  'H2O': {'H': 2, 'O': 1}
}



### Calculating Element Totals

Recall that in the previous notebook, we found that balancing a reaction requires one variable for each coefficient, 
and one equation for each element.

To start, let's write a function that will create the variables
and an equation to represent the total amount of each element in one half of the reaction.

For our example reaction:

$$H_2 + O_2 \rightarrow  H_2O$$

We need a coefficient for $ H_2 $ and $ O_2 $, as well as an equation to represent the number of hydrogen and oxygen atoms on the reactants side.
**Replace lines in the code below** to finish this function.


In [ ]:

def count_elements(reaction_half):
  # Create a list of coefficients
  coefficients = []
  # Create a dictionary that maps an element to an equation to calculate it's total on this half
  element_totals = {}
  
  for compound in reaction_half:
  
    # Create a z3 variable for each coefficient
    coefficient = False # REPLACE THIS LINE
    coefficients.append( coefficient ) 

    # Count the amount of each element in this compound
    compound_elements = reaction_half[compound]
    for element in compound_elements:
      amount_of_element = compound_elements[element]

      total = False # REPLACE THIS LINE

      if element not in element_totals:
        element_totals[element] = total
      else:
        element_totals[element] += total
  
  return coefficients, element_totals



Test your function below by confirming that it correctly creates a variable for each compound on the reactants side of the equation:


In [ ]:

coefficients, element_totals = count_elements(reactants)

print( 'Coefficients for each reactant:', coefficients )
print( 'Total of each element:', element_totals )



### Balancing Both Sides

Great! All that is left is to use this function to balance both halves of the reaction. 
Remember, in order to balance the reaction, we must ensure the amounts of an element on both sides are *equal*. 

**Replace lines in the code below** to finish the full function


In [ ]:

def create_reaction_solver(reactants, products):
  s = Solver()

  # Get the coefficients and equations for both sides
  reactants_coef, reactants_totals = count_elements(reactants)
  products_coef, products_totals = count_elements(products)

  # Ensure the amount of each element is the same on reactants and products side
  for element in reactants_totals:
    s.add( False ) # REPLACE THIS LINE

  coefficients = reactants_coef + products_coef
  
  # Ensure all coefficients are positive
  for coefficient in coefficients:
    s.add( False ) # REPLACE THIS LINE

  return s



Let's test our function with the example reaction!


In [ ]:

reactants = {
  'H2': {'H': 2},
  'O2': {'O': 2}
}

products = {
  'H2O': {'H': 2, 'O': 1}
}

s = create_reaction_solver(reactants, products)

showSolver( s ) # View the equations
print( s.check() ) # Check if solution exists


In [ ]:
print( s.model() ) # output solution


### Combustion of Propanol

Great! Let's test our function one last time with a more complex reaction, the combustion of propanol.

$$C_3H_7OH + O_2 \rightarrow CO_2 + H_2O$$

**Replace the lines in the code below** to input the above reaction into our solver.


In [ ]:

reactants = { } # REPLACE THIS LINE

products = { } # REPLACE THIS LINE

s = create_reaction_solver(reactants, products)

showSolver( s ) # View the equations
print( s.check() ) # Check if solution exists


In [ ]:
print( s.model() ) # output solution


### Congratulations! You have written a program to balance any reaction using Z3!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm